# Day 8 — Grad-CAM Explainability: iccad2

Goal: understand *what* the iccad2 baseline CNN actually looks at when making predictions, given Day 5's finding that iccad2 shows weak, near-threshold minority-class confidence (HS mean ~0.47 on test, versus a much more separated pattern on benchmarks with more positive training examples).

This notebook covers the method, two real implementation bugs found and fixed along the way (worth keeping, not just the clean final result), and the actual findings across true positives, true negatives, false positives, and false negatives.

## Method

Grad-CAM computes the gradient of the model's output prediction with respect to the last convolutional layer's activation maps (post-ReLU, pre-GAP, shape `(52, 52, 128)` for this architecture). Each of the 128 filters' gradients are spatially averaged into one importance weight, the activation maps are combined via this weighted sum, and a ReLU keeps only regions that positively contributed to the prediction. The result is upsampled to the original 224x224 resolution and overlaid on the input image.

```python
def make_gradcam_heatmap(grad_model, img_array):
    with tf.GradientTape() as tape:
        conv_out, pred = grad_model(img_array, training=False)
        target = pred[:, 0]
    grads = tape.gradient(target, conv_out)
    weights = tf.reduce_mean(grads, axis=(1, 2))
    weights_reshaped = tf.reshape(weights, (1, 1, 1, -1))
    heatmap = tf.reduce_sum(conv_out * weights_reshaped, axis=-1)
    heatmap = tf.nn.relu(heatmap)
    return heatmap, pred.numpy()[0][0]
```

## Two real bugs found and fixed

**Bug 1 -- BatchNorm/Dropout training-mode contamination.** Building `grad_model` by manually looping `x = layer(x)` over `model.layers` (needed because Keras' `.output` attribute access fails on loaded `Sequential` models -- a known limitation) initially left BatchNormalization and Dropout running in *training-mode* behavior by default. BatchNorm computed statistics from the current single-image batch (meaningless for batch size 1) instead of using its learned moving averages; Dropout randomly zeroed neurons instead of passing values through. This produced predictions wildly different from the same model's real predictions (e.g. 0.041 via `grad_model` vs the true 0.562, confirmed by cross-checking three independent prediction paths on the same file). Fix: `layer(x, training=False)` inside the loop itself, at graph-construction time -- setting `training=False` only on the outer `grad_model(img, training=False)` call was insufficient.

**Bug 2 -- silent prediction/dataframe misalignment from `os.listdir()` ordering.** `test_preds` (a cached prediction array) and a freshly-rebuilt `test_df` were assumed to correspond row-for-row, verified only by `len(test_preds) == len(test_df)`. This check passed, but the two were misaligned: `os.listdir()` does not guarantee stable ordering across separate calls, so a `test_df` rebuilt in a later session listed files in a different order than the one used when `test_preds` was originally computed. Caught by an internal-consistency check -- "false positive" examples showed a prediction of 0.000, which contradicts a false positive's own definition (prediction >= 0.5) by construction. Fix: explicitly `.sort_values("filename")` on any rebuilt file-list DataFrame, and re-save predictions together with the exact DataFrame that produced them, rather than trusting length alone as an alignment guarantee.

## Findings (post-fix, verified)

Examined across true positives, true negatives, false positives, and false negatives (3 examples each, verified against the corrected, sorted `test_preds`/`test_df` pair before interpretation).

**A real, if inconsistent, pattern: attention frequently misallocated to plain, wide horizontal bars rather than dense, jogged/cornered geometry.**
- Both false-negative examples show their single hottest region on a plain wide bar, while the actual dense jog cluster elsewhere in the same image -- the geometry Day 2's EDA flagged as plausibly hotspot-relevant -- stays comparatively cool.
- Two of three false-positive examples show the same signature: hottest attention on a wide bar, on images that are not actually hotspots.
- **However, this is not a universal rule.** One true-negative example also shows its hottest region on a bright wide bar, yet the model still correctly predicts NHS (0.000) -- meaning bar-attention alone does not deterministically flip the prediction. Something about how bar-attention is weighed against the rest of the image differs between this correctly-classified case and the error cases.

**Honest conclusion**: a subset of iccad2's errors correlate with attention misallocated toward plain wide-bar features instead of geometrically complex regions, but this is an inconsistent tendency rather than a hard, always-triggered rule. This is a more defensible finding than a single clean causal story, and worth stating with that qualification intact rather than overclaiming a fully solved explanation.

## Implications

This finding is a plausible partial explanation for iccad2's weak baseline performance (Day 5): if the model's attention is inconsistently distracted by a spurious, easy-to-detect feature (wide bars) rather than reliably focusing on the geometrically meaningful risk indicators, that would produce exactly the kind of uncertain, near-threshold predictions Day 5 observed -- separate from, and potentially not fully fixable by, class-imbalance reweighting alone. Worth testing directly once focal-loss results for iccad2 are available: does focal loss's alpha term change *what* the model attends to (a representation-learning effect), or only shift *how confidently* it reports the same attention pattern (a calibration-only effect)? Re-running this same Grad-CAM analysis on the focal-loss iccad2 model, once trained, would directly answer that.

**Next:** repeat this same TP/TN/FP/FN Grad-CAM grid on iccad3, iccad4, and iccad5 (methodology now verified, should be faster -- no bugs to rediscover) to check whether the wide-bar attention pattern is iccad2-specific or appears more broadly. iccad1 is excluded -- its baseline showed total prediction collapse with no real decision boundary to explain.